In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
from selenium.webdriver.chrome.service import Service  # Critical addition
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
import os
from pathlib import Path

LEAGUES = [
    {
        "url": "https://fbref.com/en/comps/9/Premier-League-Stats#all_stats_squads_standard",
        "name": "Premier League",
        "filename": "Premier League stats.csv"
    },
    {
        "url": "https://fbref.com/en/comps/11/Serie-A-Stats#all_stats_squads_standard",
        "name": "Serie A",
        "filename": "Serie A stats.csv"
    },
    {
        "url": "https://fbref.com/en/comps/20/Bundesliga-Stats#all_stats_squads_standard",
        "name": "Bundesliga",
        "filename": "Bundesliga stats.csv"
    },
    {
        "url": "https://fbref.com/en/comps/13/Ligue-1-Stats#all_stats_squads_standard",
        "name": "Ligue 1",
        "filename": "Ligue 1 stats.csv"
    },
    {
        "url": "https://fbref.com/en/comps/12/La-Liga-Stats#all_stats_squads_standard",
        "name": "La Liga",
        "filename": "La Liga stats.csv"
    },
    {
        "url": "https://fbref.com/en/comps/10/Championship-Stats",
        "name": "Championship",
        "filename": "Championship.csv"
    },
    {
        "url": "https://fbref.com/en/comps/18/Serie-B-Stats#all_stats_squads_standard",
        "name": "Serie B",
        "filename": "Serie B stats.csv"
    }
    # Add other leagues here...
]

def scrape_league_stats(url, league_name, filename, output_dir):
    driver = None
    try:
        options = webdriver.ChromeOptions()
        #options.add_argument('--headless')
        options.add_argument('--disable-gpu')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--window-size=1920,1080')

        # Corrected driver initialization
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )

        driver.get(url)
        print(f"Scraping {league_name}...")

        wait = WebDriverWait(driver, 30)

        # Table interaction logic
        table_container = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div#div_stats_squads_standard_for")))
        switcher = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "div.switcher")))
        driver.execute_script("arguments[0].scrollIntoView(true);", switcher)
        time.sleep(1)
        driver.execute_script("arguments[0].click();", switcher)
        time.sleep(3)

        table = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table#stats_squads_standard_for")))

        # Data extraction
        header_rows = table.find_elements(By.TAG_NAME, "thead")[0].find_elements(By.TAG_NAME, "tr")
        header_cells = header_rows[1].find_elements(By.TAG_NAME, "th")
        headers = [cell.get_attribute('data-stat') for cell in header_cells]

        tbody = table.find_element(By.TAG_NAME, "tbody")
        rows = tbody.find_elements(By.TAG_NAME, "tr")

        data = []
        for row in rows:
            cells = row.find_elements(By.CSS_SELECTOR, "th, td")
            row_data = [cell.text for cell in cells]
            if row_data:
                data.append(row_data)

        df = pd.DataFrame(data, columns=headers)
        df['league'] = league_name

        # Save file
        individual_path = os.path.join(output_dir, filename)
        df.to_csv(individual_path, index=False)
        print(f"Saved {filename}")

        return df

    except Exception as e:
        print(f"Error scraping {league_name}: {str(e)}")
        return pd.DataFrame()
    finally:
        if driver:
            driver.quit()

# Rest of the code (scrape_all_leagues, etc.) remains unchanged
def scrape_all_leagues(output_dir=None):
    if output_dir is None:
        output_dir = str(Path.home() / "Documents")
    os.makedirs(output_dir, exist_ok=True)
    
    all_data = []
    
    for league in LEAGUES:
        df = scrape_league_stats(
            url=league["url"],
            league_name=league["name"],
            filename=league["filename"],  # Pass filename here
            output_dir=output_dir
        )
        if not df.empty:
            all_data.append(df)
    
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        combined_path = os.path.join(output_dir, "Combined_Leagues_Stats.csv")
        combined_df.to_csv(combined_path, index=False)
        print(f"\nAll data combined and saved to {combined_path}")
        return combined_df
    else:
        print("No data was scraped.")
        return pd.DataFrame()

if __name__ == "__main__":
    output_dir = str(Path.home() / "Documents")
    combined_data = scrape_all_leagues(output_dir)
    print("\nSample of combined data:")
    print(combined_data.head())

Scraping Premier League...
Saved Premier League stats.csv
Scraping Serie A...
Saved Serie A stats.csv
Scraping Bundesliga...
Saved Bundesliga stats.csv
Scraping Ligue 1...
Saved Ligue 1 stats.csv
Scraping La Liga...
Saved La Liga stats.csv
Scraping Championship...
Saved Championship.csv
Scraping Serie B...
Saved Serie B stats.csv

All data combined and saved to C:\Users\DATA-JOHN\Documents\Combined_Leagues_Stats.csv

Sample of combined data:
          team players_used avg_age possession games games_starts minutes  \
0      Arsenal           24    26.5       55.8    29          319   2,610   
1  Aston Villa           28    27.7       51.4    29          319   2,610   
2  Bournemouth           28    25.8       47.0    29          319   2,610   
3    Brentford           27    26.6       48.2    29          319   2,610   
4     Brighton           30    25.6       51.8    29          319   2,610   

  minutes_90s goals assists  ... assists_per90 goals_assists_per90  \
0        29.0    51  